In [5]:
import pandas as pd
import os
import io
import re
import paramiko
from io import BytesIO
from ftplib import FTP
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
# def ConnectFTP(server, username, password):
#     """
#     Descripcion: Esta funcion establece una conexion FTP con el servidor especificado.
#     Parametros:
#     server (str): La direccion del servidor FTP.
#     username (str): El nombre de usuario para la conexion FTP.
#     password (str): La contrasena para la conexion FTP.
#     Retorna: Un objeto FTP conectado al servidor.
#     """
#     ftp = FTP(timeout=60)                
#     ftp.connect(server, 21)               
#     ftp.login(user=username, passwd=password)

#     ftp.set_pasv(True)                    
#     ftp.voidcmd("TYPE I")                 

#     print(f"Conectado a {server}")
#     return ftp

# def UploadCsvToFTP(df, path):
#     """
#     Descripcion: Esta funcion sube un DataFrame de pandas como un archivo CSV a un servidor FTP.
#     Parametros:
#     df (DataFrame): El DataFrame que se desea subir.
#     path (str): La ruta en el servidor FTP donde se guardará el archivo CSV.
#     Retorna: None
#     """
#     csv_buffer = io.BytesIO()
#     df.to_csv(csv_buffer, index=False, encoding='utf-8')
#     csv_buffer.seek(0)
#     ftp = ConnectFTP(os.getenv('ftp_server'),os.getenv('ftp_user'),os.getenv('ftp_password'))
#     # remote_path = f"/pruebas/csv/{path}"
#     ftp.storbinary(f"STOR {path}", csv_buffer)
#     ftp.quit()
#     print("Archivo subido correctamente:", path)

# def ReadCsvFromFTP(remote_file_path):
#     '''
#     Descripcion: Esta funcion lee un archivo CSV desde un servidor FTP y lo carga en un DataFrame de pandas.
#     Parametros:
#     ftp (FTP): Un objeto FTP conectado al servidor.
#     remote_file_path (str): La ruta del archivo CSV en el servidor FTP.
#     Retorna: Un DataFrame de pandas que contiene los datos del archivo CSV.
#     '''
#     ftp = ConnectFTP(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password'))
#     with BytesIO() as bio:
#         ftp.retrbinary(f'RETR {remote_file_path}', bio.write)
#         bio.seek(0)
#         df = pd.read_csv(bio)
#     return df

def ConnectSFTP(server, username, password, port=22):
    """
    Establece una conexión SFTP mediante SSH.

    Retorna:
        ssh: cliente SSH necesario para cerrar correctamente la conexión.
        sftp: cliente utilizado para navegar y transferir archivos.
    """
    ssh = paramiko.SSHClient()
    ssh.load_system_host_keys()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    ssh.connect(
        hostname=server,
        port=port,
        username=username,
        password=password,
        timeout=60,
    )

    sftp = ssh.open_sftp()

    print(f"Conectado mediante SFTP a {server}:{port}")
    return ssh, sftp

def UploadCsvToSFTP(df, path):
    """
    Convierte un DataFrame a CSV en memoria y lo sube mediante SFTP.
    """
    csv_buffer = io.BytesIO()
    df.to_csv(
        csv_buffer,
        index=False,
        encoding="utf-8"
    )
    csv_buffer.seek(0)
    ssh, sftp = ConnectSFTP(
        os.getenv("ftp_server"),
        os.getenv("ftp_user"),
        os.getenv("ftp_password")
    )
    # remote_path = f"riecs/pruebas2/csv/{path}"
    try:
        sftp.putfo(csv_buffer, path)
        print("Archivo subido correctamente:", path)
    finally:
        sftp.close()
        ssh.close()

def ReadCsvFromSFTP(remote_file_path):
    """
    Lee un archivo CSV desde un servidor SFTP y lo carga
    en un DataFrame de pandas.

    Parámetros:
        remote_file_path (str):
            Ruta completa del archivo CSV dentro del servidor.

    Retorna:
        pandas.DataFrame:
            DataFrame con el contenido del archivo CSV.
    """
    ssh, sftp = ConnectSFTP(
        os.getenv("ftp_server"),
        os.getenv("ftp_user"),
        os.getenv("ftp_password")
    )
    try:
        with sftp.open(remote_file_path, mode="rb") as remote_file:
            df = pd.read_csv(remote_file)
        return df
    finally:
        sftp.close()
        ssh.close()

def Edad_Group(df):
    list_groups = ['Alcohol','Tabaco','Marihuana','Cocaína','Metanfetaminas','Opioides','Inhalables', 'Alucinógenos','Medicamentos','Nuevas Sustancias Psicoactivas','Otras Sustancias', 'ProblemasDeSaludMental' ,'Ninguna'] 
    list_cols_sust = [col for col in df.columns if col.startswith('SustanciaId')]
    list_edad = [col for col in df.columns if col.startswith('EdadInicio')]
    for group in list_groups:
        mask = df[list_cols_sust].eq(group)
        edades = df[list_edad].where(mask.values)
        df[f'EdadInicio_{group}'] = edades.min(axis=1)
    return df

def Sust_data(df):
    lista_sustancias = [
    'Tabaco', 'Nicotina en dispositivo electrónico', 'Bebidas Fermentadas',
    'Bebidas Destiladas', 'Alcohol puro (alcohol del 96)', 'Mariguana estándar',
    'Mariguana modificada, con alto nivel de THC', 'Hachís - Resina',

    'Hachís - Aceite o miel', 'Mariguana sintética',
    'Medicamentos con compuestos cannábicos', 'Cocaína en polvo blanco',
    'Crack', 'Basuco (pasta base de cocaína)',
    'Otras presentaciones de cocaína', 'Metanfetaminas(cristal)',

    'Metanfetaminas(hielo)', 'MetanfetaminasMetas', 'Sales de baño',
    'Anfetaminas y derivados', 'Anorexígenos',
    'Drogas de diseño con efectos estimulantes', 'Otros estimulantes',
    'Solventes o removedores',

    'Pegamentos', 'Esmaltes y pinturas', 'Gasolinas y combustibles',
    'Otros inhalantes', 'LSD', 'LSA', 'Hongos alucinógenos',
    'Peyote (mescalina)',

    'Salvia divinorum u otras variedades de salvia', 'Ololiuhqui',
    'Floripondio', 'Ayahuasca',
    'Otras plantas con efectos alucinógenos o disociativos',
    'Fenciclidina (PCP, polvo de ángel)', 'Análogos del PCP', 'Ketamina',

    'DMT', 'Triptaminas psicodélicas derivadas y análogas del DMT',
    'Grupo 2Cx', 'Grupo DOx', 'Grupo NBOMe',
    'Otras drogas de diseño con efectos psicodélicos',
    'Otras sustancias con efectos psicodélicos', 'Éxtasis (MDMA)',

    'Drogas de diseño con efectos similares al éxtasis',
    'Piperazinas con efectos similares al éxtasis', 'Aminoindanos',
    'Benzodiacepinas y sustancias relacionadas', 'Rohypnol',
    'GHB o similares', 'Sedantes hipnóticos', 'Otros depresores',

    'Opio', 'Morfina(uso fuera de prescripción)',
    'Codeína(uso fuera de prescripción)', 'Heroína blanca',
    'Heroína negra', 'Oxicodona',
    'Opiodes sintéticos (uso fuera de prescripción)',
    'Fentanilo y análogos (uso fuera de prescripción)',

    'Dextrometorfano (uso fuera de prescripción)',
    'Derivados opiodes combinados con analgésicos', 'Desomorfina',
    'Anticolinérgicos', 'Antiespasmódicos', 'Antiparkinsonianos',
    'Anticonvulsivos', 'Antidepresivos',

    'Antipsicóticos', 'Antihistamínicos',
    'Otras sustancias de abuso con utilidad médica', 'Nitritos',
    'Anestésicos', 'Bebidas energizantes', 'Bebidas inteligentes',
    'Esteroides anabólicos',

    'Sustancias no clasificadas', 'Sustancias sin especificar',
    'Problemas de salud mental', 'Ninguna'
]
    
    list_cols_sust = [col for col in df.columns if col.startswith('SustanciaId')]
    for sust in lista_sustancias:
        df[sust] = df[list_cols_sust].eq(sust).any(axis=1).astype(int)
    return df

def Edad_Sust(df):
    lista_sustancias = [
    'Tabaco', 'Nicotina en dispositivo electrónico', 'Bebidas Fermentadas',
    'Bebidas Destiladas', 'Alcohol puro (alcohol del 96)', 'Mariguana estándar',
    'Mariguana modificada, con alto nivel de THC', 'Hachís - Resina',

    'Hachís - Aceite o miel', 'Mariguana sintética',
    'Medicamentos con compuestos cannábicos', 'Cocaína en polvo blanco',
    'Crack', 'Basuco (pasta base de cocaína)',
    'Otras presentaciones de cocaína', 'Metanfetaminas(cristal)',

    'Metanfetaminas(hielo)', 'MetanfetaminasMetas', 'Sales de baño',
    'Anfetaminas y derivados', 'Anorexígenos',
    'Drogas de diseño con efectos estimulantes', 'Otros estimulantes',
    'Solventes o removedores',

    'Pegamentos', 'Esmaltes y pinturas', 'Gasolinas y combustibles',
    'Otros inhalantes', 'LSD', 'LSA', 'Hongos alucinógenos',
    'Peyote (mescalina)',

    'Salvia divinorum u otras variedades de salvia', 'Ololiuhqui',
    'Floripondio', 'Ayahuasca',
    'Otras plantas con efectos alucinógenos o disociativos',
    'Fenciclidina (PCP, polvo de ángel)', 'Análogos del PCP', 'Ketamina',

    'DMT', 'Triptaminas psicodélicas derivadas y análogas del DMT',
    'Grupo 2Cx', 'Grupo DOx', 'Grupo NBOMe',
    'Otras drogas de diseño con efectos psicodélicos',
    'Otras sustancias con efectos psicodélicos', 'Éxtasis (MDMA)',

    'Drogas de diseño con efectos similares al éxtasis',
    'Piperazinas con efectos similares al éxtasis', 'Aminoindanos',
    'Benzodiacepinas y sustancias relacionadas', 'Rohypnol',
    'GHB o similares', 'Sedantes hipnóticos', 'Otros depresores',

    'Opio', 'Morfina(uso fuera de prescripción)',
    'Codeína(uso fuera de prescripción)', 'Heroína blanca',
    'Heroína negra', 'Oxicodona',
    'Opiodes sintéticos (uso fuera de prescripción)',
    'Fentanilo y análogos (uso fuera de prescripción)',

    'Dextrometorfano (uso fuera de prescripción)',
    'Derivados opiodes combinados con analgésicos', 'Desomorfina',
    'Anticolinérgicos', 'Antiespasmódicos', 'Antiparkinsonianos',
    'Anticonvulsivos', 'Antidepresivos',

    'Antipsicóticos', 'Antihistamínicos',
    'Otras sustancias de abuso con utilidad médica', 'Nitritos',
    'Anestésicos', 'Bebidas energizantes', 'Bebidas inteligentes',
    'Esteroides anabólicos',

    'Sustancias no clasificadas', 'Sustancias sin especificar',
    'Problemas de salud mental', 'Ninguna'
]
    list_cols_sust = [col for col in df.columns if col.startswith('SustanciaId')]
    list_edad = [col for col in df.columns if col.startswith('EdadInicio')]
    for sust in lista_sustancias:
        mask = df[list_cols_sust].eq(sust)
        edades = df[list_edad].where(mask.values)
        df[f'EdadInicio_{sust}'] = edades.min(axis=1)
    return df

def Elim_Cols (df):
    cols_eliminar = [
    col for col in df.columns
    if re.match(r'^EdadInicio\d+$', col)
]
    df = df.drop(columns=cols_eliminar)
    return df

In [7]:
def main():
    try:
        list_ftp_groups = ['EntrevistaInicialUniverso','EntrevistaInicialDrogasLegalesIlegales','EntrevistaInicialDrogasIlegales']
        for name in list_ftp_groups:
            df = ReadCsvFromSFTP(f'riecs/pruebas2/csv/Recodificacion/{name}.csv')
            # df = Group_data(df)
            df = Edad_Group(df)
            df = Elim_Cols(df)
            UploadCsvToSFTP(df, f'riecs/pruebas2/csv/Recodificacion/EdadInicio/{name}.csv')
    except Exception as e:
        print("4)_EdadInicio_Recod / Error en el proceso principal (Grupos):", e)
        return
    
    try:
        list_ftp_groups = ['EntrevistaInicialUniversoSust', 'EntrevistaInicialDrogasLegalesIlegalesSust', 'EntrevistaInicialDrogasIlegalesSust']
        for name in list_ftp_groups:
            df_sust = ReadCsvFromSFTP(f'riecs/pruebas2/csv/Recodificacion/{name}.csv')
            # df = Sust_data(df)
            df_sust = Edad_Sust(df_sust)
            df_sust = Elim_Cols(df_sust)
            UploadCsvToSFTP(df_sust, f'riecs/pruebas2/csv/Recodificacion/EdadInicio/{name}.csv')
    except Exception as e:
        print("4)_EdadInicio_Recod / Error en el proceso principal (Sustancias):", e)
        return
    return df, df_sust

In [8]:
df, dfsust = main()

Conectado mediante SFTP a 192.168.62.150:22


C:\Users\franc\AppData\Local\Temp\ipykernel_27148\1612672229.py:121: DtypeWarning: Columns (31,50,51,52,53,54,70,71,72,73,74,75,76,77,78,79,80,81,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,322,330,344,345,346,347,348,349,366,367,370,371,372,373,374,375,424,425,426) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(remote_file)


Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EdadInicio/EntrevistaInicialUniverso.csv
Conectado mediante SFTP a 192.168.62.150:22


C:\Users\franc\AppData\Local\Temp\ipykernel_27148\1612672229.py:121: DtypeWarning: Columns (31,50,51,52,53,54,70,71,72,73,74,75,76,77,78,79,80,81,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,322,330,344,345,346,347,348,349,366,367,370,371,372,373,374,375,424,425,426) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(remote_file)


Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EdadInicio/EntrevistaInicialDrogasLegalesIlegales.csv
Conectado mediante SFTP a 192.168.62.150:22


...

Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EdadInicio/EntrevistaInicialUniversoSust.csv
Conectado mediante SFTP a 192.168.62.150:22


C:\Users\franc\AppData\Local\Temp\ipykernel_27148\1612672229.py:121: DtypeWarning: Columns (31,50,51,52,53,54,70,71,72,73,74,75,76,77,78,79,80,81,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,322,330,344,345,346,347,348,349,366,367,370,371,372,373,374,375,424,425,426) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(remote_file)


Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EdadInicio/EntrevistaInicialDrogasLegalesIlegalesSust.csv
Conectado mediante SFTP a 192.168.62.150:22


C:\Users\franc\AppData\Local\Temp\ipykernel_27148\1612672229.py:121: DtypeWarning: Columns (31,50,51,52,53,54,70,71,72,73,74,75,76,77,78,79,80,81,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,322,330,344,345,346,347,348,349,366,367,370,371,372,373,374,375,424,425,426) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(remote_file)


Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/Recodificacion/EdadInicio/EntrevistaInicialDrogasIlegalesSust.csv
